In [1]:
!pip install transformers==4.57.1 datasets==4.0.0 peft==0.17.0 trl==0.21.0 accelerate==1.10.0 bitsandbytes==0.48.1
import os

!pip install "git+https://github.com/red-hat-data-services/mlflow@rhoai-3.3"


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/red-hat-data-services/mlflow (to revision rhoai-3.3) to /tmp/pip-req-build-xzoee5jw
  Running command git clone --filter=blob:none --quiet https://github.com/red-hat-data-services/mlflow /tmp/pip-req-build-xzoee5jw
  Running command git checkout -b rhoai-3.3 --track origin/rhoai-3.3
  Switched to a new branch 'rhoai-3.3'
  branch 'rhoai-3.3' set up to track 'origin/rhoai-3.3'.
  Resolved https://github.com/red-hat-data-services/mlflow to commit c0f86528f988088bc07be8acc0db50074acee0f4
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os

os.environ["MLFLOW_TRACKING_TOKEN"] = "sha256~kE5RlUe5Lm3KFi41teRVxF4nAN_VdYc8lkYIa-I8yVc"
os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
os.environ["MLFLOW_WORKSPACE"] = "decoder-sft"
os.environ["MLFLOW_RUN_NAME"] = "decoder-sft-run"
os.environ["MLFLOW_TRACKING_HEADERS"] = "{\"X-MLFLOW-WORKSPACE\": \"decoder-sft\", \"Content-Type\": \"application/json\"}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "decoder-sft"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
print("Environment variables set!")

Environment variables set!


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import shutil
import os
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from datetime import datetime
import json
import mlflow
from transformers import TrainerCallback

/opt/app-root/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class MLflowLineageCallback(TrainerCallback):
    """
    custom callback to inject S3 dataset lineage and LoRA hyperparameters 
    into the MLflow run automatically created by Hugging Face Trainer
    """
    def __init__(self, custom_args, peft_config, train_ds, val_ds, test_ds):
        self.custom_args = custom_args
        self.peft_config = peft_config
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.test_ds = test_ds

    def on_train_begin(self, args, state, control, **kwargs):
        # only execute on the main process (Rank 0) 
        if state.is_world_process_zero:
            print("Injecting S3 data lineage into Hugging Face's active MLflow run...")
            
            # HF automatically logs learning_rate, batch_size, epochs, etc.
            # so we omit them here to prevent MLflow "duplicate param" errors
            mlflow.log_params({
                "model_name": self.custom_args.model_name,
                "data_path": self.custom_args.data_path,
                "lora_r": self.peft_config.r,
                "lora_alpha": self.peft_config.lora_alpha,
                "lora_dropout": self.peft_config.lora_dropout,
            })
            
            # convert datasets and log them directly into HF's run
            ds_train = mlflow.data.from_pandas(
                self.train_ds.to_pandas(), source=self.custom_args.dataset_source, name="intent_train"
            )
            ds_val = mlflow.data.from_pandas(
                self.val_ds.to_pandas(), source=self.custom_args.dataset_source, name="intent_val"
            )
            ds_test = mlflow.data.from_pandas(
                self.test_ds.to_pandas(), source=self.custom_args.dataset_source, name="intent_test"
            )
            
            mlflow.log_input(ds_train, context="training")
            mlflow.log_input(ds_val, context="evaluation")
            mlflow.log_input(ds_test, context="test")

In [5]:
def format_messages_for_training(row: dict, base_model_id: str) -> dict:
    """
    translates the enterprise data schema into a model agnostic chat template for SFTTrainer
    """
    messages = []
    system_prompt = (
        "You are an AI intent classifier. Analyze the conversation history "
        "and the latest user message. You must output only a valid JSON object "
        "containing the predicted 'intent'."
    )
    
    model_name = base_model_id.lower()
    
    # handle model-specific system prompt support
    if "qwen" in model_name or "phi" in model_name or "deepseek" in model_name:
        messages.append({"role": "system", "content": system_prompt})
    elif "gemma" in model_name:
        pass 
    else:
        messages.append({"role": "system", "content": system_prompt})

    # process session history
    for turn in row.get("session_history", []):
        print (turn)
        hf_role = "assistant" if turn.get("role") == "assistant" else "user"
        messages.append({"role": hf_role, "content": turn.get("content", "")})
        
    # process current message
    current_msg = row.get("message", "")
    if "gemma" in model_name and len(messages) == 0:
        current_msg = f"{system_prompt}\n\n{current_msg}"
        
    messages.append({"role": "user", "content": current_msg})
    
    # critical for training: append the target output so the model learns to generate it
    target_output = json.dumps({"intent": row.get("intent", "UNKNOWN")})
    messages.append({
        "role": "assistant",
        "content": target_output
    })
    
    return {"messages": messages}

In [6]:
# test the formatter with a sample row
sample_row = {
    "message": "I want to change my shipping address",
    "intent": "UPDATE_SHIPPING",
    "session_history": [
        {"role": "assistant", "content": "Hello! How can I help you today?"},
        {"role": "user", "content": "I placed an order yesterday"}
    ]
}

# test with Qwen (supports system prompt)
result = format_messages_for_training(sample_row, "Qwen/Qwen2.5-7B-Instruct")
print("=== Qwen formatted output ===")
for msg in result["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}")

print()

# test with Gemma (no system prompt, prepended to first user message)
sample_no_history = {"message": "check my balance", "intent": "CHECK_BALANCE", "session_history": []}
result_gemma = format_messages_for_training(sample_no_history, "google/gemma-2b")
print("=== Gemma formatted output (no history) ===")
for msg in result_gemma["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}")

{'role': 'assistant', 'content': 'Hello! How can I help you today?'}
{'role': 'user', 'content': 'I placed an order yesterday'}
=== Qwen formatted output ===
  [system]: You are an AI intent classifier. Analyze the conversation history and the latest
  [assistant]: Hello! How can I help you today?
  [user]: I placed an order yesterday
  [user]: I want to change my shipping address
  [assistant]: {"intent": "UPDATE_SHIPPING"}

=== Gemma formatted output (no history) ===
  [user]: You are an AI intent classifier. Analyze the conversation history and the latest
  [assistant]: {"intent": "CHECK_BALANCE"}


In [7]:
model_name = "Qwen/Qwen2.5-3B-Instruct"
data_path = "./mnt/data/train.json"      # path to JSON training data
output_dir = "./mnt/data/adapters"
dataset_source = ""                               # s3 uri for MLflow lineage
run_name = "lora-finetune-run"

# multi GPU rank: set by torchrun, defaults to 0 for single GPU
local_rank = int(os.environ.get("LOCAL_RANK", 0))
print(f"local_rank: {local_rank}")

# rank 0 housekeeping: clean old output to avoid adapter corruption
if local_rank == 0:
    if os.path.exists(output_dir):
        print(f"cleaning existing output dir: {output_dir}")
        shutil.rmtree(output_dir)

local_rank: 0
cleaning existing output dir: ./mnt/data/adapters


In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"quantization config: {bnb_config}")
print(f"loading model: {model_name}...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, 
    device_map={"":local_rank},
    torch_dtype=torch.float16, 
    trust_remote_code=True 
)

# CRITICAL: Force the base config to fp16 so PEFT doesn't secretly initialize LoRA weights in bfloat16
model.config.torch_dtype = torch.float16
model.config.use_cache = False


tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# override with a TRL-compatible ChatML template that includes generation markers
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'user' %}"
    "<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
    "{% elif message['role'] == 'assistant' %}"
    "<|im_start|>assistant\n{% generation %}{{ message['content'] }}{% endgeneration %}<|im_end|>\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{% endif %}"
)

print(f"pad_token: {tokenizer.pad_token}")
print(f"vocab size: {tokenizer.vocab_size}")


data_path = "./mnt/data/train.json"
dataset = load_dataset("json", data_files=data_path, split="train")

# 80/10/10 split
split_1 = dataset.train_test_split(test_size=0.2, seed=42)
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = split_1["train"]
val_dataset = split_2["train"]
test_dataset = split_2["test"]

print(f"train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_dataset)}")

# quarantine the test set so it is never touched during training
if local_rank == 0:
    print("saving quarantined test set to ./mnt/data/test_split.jsonl...")
    test_dataset.to_json("./mnt/data/test_split.jsonl")

# apply the chat template transformation to train and val sets
print("formatting dataset for the specific model...")

train_dataset = train_dataset.map(
    lambda row: format_messages_for_training(row, model_name), 
    remove_columns=["message", "intent", "session_history"]
)

val_dataset = val_dataset.map(
    lambda row: format_messages_for_training(row, model_name), 
    remove_columns=["message", "intent", "session_history"]
)
# remove_columns drops the original fields after they have been folded
# into the 'messages' column, freeing memory

print(f"train columns after formatting: {train_dataset.column_names}")
print(f"sample formatted row:")
print(json.dumps(train_dataset[0], indent=2))
    


r = 16
peft_config = LoraConfig(
    r=r,
    lora_alpha=(2 * r),
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

training_args = SFTConfig(
    output_dir=output_dir,
    max_length=1024,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=25,
    num_train_epochs=3,
    max_grad_norm=0.3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    bf16=False, # Explicitly deny bfloat16 to prevent T4 crashes
    push_to_hub=False,
    packing=False,
    assistant_only_loss=True, # <-- Critical for intent classification

    # mlflow tracking
    report_to="mlflow",
    run_name=run_name,

    # evaluation and checkpointing
    eval_strategy="steps",
    eval_steps=0.1,
    save_strategy="steps",
    save_steps=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Memory optimizations
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}, 
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
)

from types import SimpleNamespace
args_ns = SimpleNamespace(
    model_name=model_name,
    data_path=data_path,
    dataset_source=dataset_source
)

lineage_callback = MLflowLineageCallback(
    custom_args=args_ns,
    peft_config=peft_config,
    train_ds=train_dataset,
    val_ds=val_dataset,
    test_ds=test_dataset
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config, 
    processing_class=tokenizer,
    callbacks=[lineage_callback]
)

print("starting training...")
trainer.train()

`torch_dtype` is deprecated! Use `dtype` instead!


quantization config: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

loading model: Qwen/Qwen2.5-3B-Instruct...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]/opt/app-root/lib64/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 2/2 [00:22<00:00, 11.30s/it]


pad_token: <|im_end|>
vocab size: 151643
train: 8 | val: 1 | test: 1
saving quarantined test set to ./mnt/data/test_split.jsonl...


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 172.62ba/s]

...

Step,Training Loss,Validation Loss
1,No log,4.303426
2,No log,2.962828
3,No log,1.963429
4,No log,1.729706
5,No log,1.550506
6,No log,1.511494


/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

🏃 View run lora-finetune-run at: https://mlflow.redhat-ods-applications.svc.cluster.local:8443/#/workspaces/decoder-sft/experiments/5/runs/db2b56d0eaa9473ba55781d3983d30fc
🧪 View experiment at: https://mlflow.redhat-ods-applications.svc.cluster.local:8443/#/workspaces/decoder-sft/experiments/5


/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


TrainOutput(global_step=6, training_loss=2.6147286097208657, metrics={'train_runtime': 26.8833, 'train_samples_per_second': 0.893, 'train_steps_per_second': 0.223, 'total_flos': 72294136086528.0, 'train_loss': 2.6147286097208657})